## As per my understanding:
- **Generators** are an easy way to create iterator objects.
-They are functions that use yield instead of (or alongside) return.
- They execute incrementally rather than running completely in one go.
- They remember their execution state between calls.
- You use next() to retrieve the next yielded value.
- You cannot go backwards through a generator once it has advanced.

In [ ]:
def generator_fun():
    yield 1
    yield 2
    yield 3
    
gen = generator_fun()   # Note: Generator function is CREATED, NOT function called as in normal function

print(next(gen))   # next actually starts executing
print(next(gen))
print(next(gen))
    

1
2
3


In [4]:
print(next(gen))

StopIteration: 

In [ ]:
# Can do generator comprehension just like list comprehension but with parenthsis

listt = [x for x in range(10000)]

numbers = (x for x in range(10000))   # notice parenthesis instead of square brackets
next(numbers)
next(numbers)

1

The list stores all one million numbers. The generator produces each number only when requested.

## Formal Definition
A generator is a special type of iterator created using a function containing yield. Unlike normal functions, generator functions return a generator object without immediately executing their body. They execute **lazily**, yielding values one at a time, while preserving their execution state between successive calls to next(). This makes them *memory-efficient* for processing large or potentially infinite sequences.

In [ ]:
from typing import Iterable, Iterator, List


def batch_iterator(chunks: Iterable[str], batch_size: int) -> Iterator[List[str]]:
    """Yields fixed-size batches lazily without loading all chunks into memory."""
    if batch_size <= 0:
        raise ValueError("batch_size must be greater than 0")

    batch: List[str] = []
    for chunk in chunks:
        batch.append(chunk)
        if len(batch) == batch_size:
            yield batch
            batch = [] #empty batch after yield

    # Handle the final partial batch
    if batch:
        yield batch


# --- Simulated RAG Ingestion Pipeline ---


def fake_embed_api(texts: List[str]) -> List[List[float]]:
    """Simulates calling an embedding model (e.g., OpenAI, HuggingFace)."""
    return [[float(len(text)), 0.1, 0.2] for text in texts]


# Imagine this comes from reading a massive file line-by-line (an iterator)
document_chunks = (f"Document chunk #{i}" for i in range(1, 8))

# Process 3 chunks at a time
for batch_num, batch in enumerate(batch_iterator(document_chunks, batch_size=3), start=1):
    vectors = fake_embed_api(batch)
    print(f"Batch {batch_num}: Embedded {len(batch)} chunks -> {batch}")
    # Next step: upsert `vectors` directly to a vector database (e.g., Pinecone, Chroma)


Batch 1: Embedded 3 chunks -> ['Document chunk #1', 'Document chunk #2', 'Document chunk #3']
Batch 2: Embedded 3 chunks -> ['Document chunk #4', 'Document chunk #5', 'Document chunk #6']
Batch 3: Embedded 1 chunks -> ['Document chunk #7']


### How `batch_iterator` Keeps Memory Low in the above example

Only `batch_size` (e.g., `3`) items are held in active memory at once—the remaining chunks are neither created nor read until the next batch is requested.

#### Step-by-Step Execution Flow
1. **Pulling one by one:** `batch_iterator` requests one chunk at a time from `document_chunks` (`#1`, `#2`, `#3`) and appends each to `batch`.
2. **Pausing at `yield`:** Once `len(batch) == 3`, `yield batch` passes that 3-item list to the `for` loop and **freezes** `batch_iterator` at that exact line. Chunks `#4` through `#7` do not exist in memory yet.
3. **Processing & discarding:** The loop runs `fake_embed_api(batch)` on those 3 chunks. Once that iteration finishes, Python requests the next batch from `batch_iterator`.
4. **Resuming & resetting:** `batch_iterator` resumes on the very next line (`batch = []`), dropping the reference to the previous 3 chunks (freeing their memory) before pulling `#4`, `#5`, and `#6`.

> **Key Nuance:** Each individual batch yielded (`['Document chunk #1', 'Document chunk #2', 'Document chunk #3']`) is a concrete `list` in memory, while `batch_iterator(...)` itself is the lazy **iterator** producing those small lists one at a time.
---

### Why Generators Matter in AI Engineering

A **generator** is a function using `yield` (or a generator expression) that produces values lazily one at a time while automatically preserving local state between steps.

* **Constant memory footprint ($O(1)$ RAM):** Stream terabytes of training data, logs, or embeddings without loading entire datasets into memory.
* **Lower latency (Time-to-First-Token):** Deliver partial LLM outputs immediately as tokens arrive instead of blocking until full generation finishes.
* **Composable data pipelines:** Chain cleaning, chunking, filtering, and batching steps together without creating intermediate lists at each stage.
* **Fault tolerance & early exit:** Stop execution mid-stream (e.g., when a guardrail triggers or top-$k$ matches are found) without wasting compute or API credits on remaining items.

---

### Core AI Engineering Use Cases

* **LLM Token & Event Streaming:** Yielding text chunks or Server-Sent Events (SSE) incrementally from model APIs to UIs (`async def` + `yield` in FastAPI/LangChain).
* **RAG Data Ingestion & Chunking:** Reading large PDFs, web scrapes, or JSONL files line-by-line, splitting them into chunks, and yielding fixed-size embedding batches.
* **Model Training Data Loaders:** Streaming, shuffling, and augmenting image/text mini-batches from disk or cloud storage on the fly so GPUs stay fed without RAM overflow.
* **Agent & LangGraph Step Tracing:** Yielding intermediate state updates, tool calls, and reasoning steps as an agent executes multi-step workflows.
* **Evaluation & Synthetic Data Pipelines:** Streaming golden dataset rows through LLM-as-a-judge evaluators and writing results incrementally to disk.


# Summary
---

# Iterators and Generators — Theory

## 1. What is an Iterable?
An object that can return an iterator using `iter()`.

## 2. What is an Iterator?
An object implementing `__iter__()` and `__next__()`.

## 3. What is a Generator?
A special iterator created using a generator function or expression.

## 4. Iterable vs Iterator
- An iterable produces an iterator.
- An iterator produces values.

## 5. `return` vs `yield`
- `return` terminates a function.
- `yield` suspends execution and preserves the function's state.

## 6. What is Lazy Evaluation?
Values are generated only when requested.

## 7. What is StopIteration?
An exception signalling that an iterator is exhausted.

## 8. Can an Iterator Be Reused?
Usually not after exhaustion. Obtain a fresh iterator from a reusable iterable.

## 9. Why Use Generators?
- Memory efficiency.
- Lazy processing.

## 10. Generator vs List
- A list stores all elements.
- A generator produces elements on demand.